In [2]:
import json
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains import LLMChain
from langchain.schema import BaseOutputParser
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

from dotenv import load_dotenv
load_dotenv()

False

In [3]:
import os
from langchain.chat_models import ChatOpenAI

os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

llm = ChatOpenAI(
    model="mistralai/ministral-8b-2512",
    temperature=0.3,
    max_tokens=4000,
    openai_api_key=os.environ.get("OPENAI_API_KEY"),
    openai_api_base=os.environ.get("OPENAI_API_BASE")
)

print("✅ OpenRouter LLM configured successfully!")
print(f"🤖 Using model: {llm.model_name}")

✅ OpenRouter LLM configured successfully!
🤖 Using model: mistralai/ministral-8b-2512


In [4]:
def load_reviewer_data(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Load reviewer data from multiple repositories
reviewer_data_paths = {
    'preact': r'PR data for Reviewers/preactjs_preact_reviewer_data_train.json'
}

all_reviewer_data = {}
total_prs = 0

print("📁 Loading reviewer data from multiple repositories...")
print("=" * 50)

for repo_name, file_path in reviewer_data_paths.items():
    if os.path.exists(file_path):
        print(f"📂 Loading {repo_name} reviewer data from: {file_path}")
        data = load_reviewer_data(file_path)
        
        if data and 'pr_reviewer_data' in data:
            all_reviewer_data[repo_name] = data
            pr_count = len(data['pr_reviewer_data'])
            total_prs += pr_count
            print(f"   ✅ Loaded {pr_count} PRs from {repo_name}")
        else:
            print(f"   ❌ Failed to load {repo_name} data")
            all_reviewer_data[repo_name] = {'pr_reviewer_data': []}
    else:
        print(f"   ⚠️  File not found: {file_path}")
        all_reviewer_data[repo_name] = {'pr_reviewer_data': []}

print("=" * 50)
print(f"📊 Total loaded: {total_prs} PRs from {len([k for k in all_reviewer_data if all_reviewer_data[k]['pr_reviewer_data']])} repositories")

combined_reviewer_data = {'pr_reviewer_data': []}
for repo_name, data in all_reviewer_data.items():
    for pr in data.get('pr_reviewer_data', []):
        pr['repo_name'] = repo_name
        combined_reviewer_data['pr_reviewer_data'].append(pr)

reviewer_data = combined_reviewer_data
print(f"🔗 Combined dataset: {len(reviewer_data['pr_reviewer_data'])} total PRs for analysis")

📁 Loading reviewer data from multiple repositories...
📂 Loading preact reviewer data from: PR data for Reviewers/preactjs_preact_reviewer_data_train.json
   ✅ Loaded 353 PRs from preact
📊 Total loaded: 353 PRs from 1 repositories
🔗 Combined dataset: 353 total PRs for analysis


In [5]:
def analyze_reviewer_activity(reviewer_data):
    if not reviewer_data or 'pr_reviewer_data' not in reviewer_data:
        return {}
    
    reviewer_stats = {}
    
    for pr in reviewer_data['pr_reviewer_data']:
        for review in pr.get('reviews', []):
            username = review.get('reviewer_username')
            if username not in reviewer_stats:
                reviewer_stats[username] = {
                    'total_reviews': 0,
                    'total_comments': 0,
                    'total_prs': 0,
                    'repos': set(),
                    'review_details': []
                }
            
            reviewer_stats[username]['total_reviews'] += 1
            reviewer_stats[username]['repos'].add(pr.get('repo_name', ''))
            reviewer_stats[username]['review_details'].append({
                'pr_number': pr.get('pr_number'),
                'title': pr.get('title', ''),
                'description': pr.get('description', ''),
                'review_state': review.get('state'),
                'review_body': review.get('body', ''),
                'labels': pr.get('labels', []),
                'additions': pr.get('additions', 0),
                'deletions': pr.get('deletions', 0),
                'changed_files_count': pr.get('changed_files_count', 0)
            })
        
        for comment in pr.get('review_comments', []):
            username = comment.get('reviewer_username')
            if username in reviewer_stats:
                reviewer_stats[username]['total_comments'] += 1
    
    for username in reviewer_stats:
        reviewer_stats[username]['repos'] = list(reviewer_stats[username]['repos'])
        reviewer_stats[username]['total_prs'] = len(set(review['pr_number'] for review in reviewer_stats[username]['review_details']))
    
    return reviewer_stats

if reviewer_data:
    reviewer_stats = analyze_reviewer_activity(reviewer_data)
    print(f"📊 Analyzed {len(reviewer_stats)} reviewers")
else:
    reviewer_stats = {}

📊 Analyzed 43 reviewers


In [6]:
class ReviewerProfile(BaseModel):
    reviewer_name: str = Field(description="Reviewer's username")
    experience_level: str = Field(description="Estimated experience level (Junior/Mid/Senior/Lead)")
    primary_skills: List[str] = Field(description="Top JavaScript-related skills inferred from review evidence")
    programming_languages: List[str] = Field(description="Programming languages expertise (JavaScript-focused)")
    summary: str = Field(description="Overall JavaScript reviewer profile summary")
    identified_skills: Dict[str, List[str]] = Field(
        description="LLM-identified skill categories and specific skills (e.g., 'Backend Frameworks': ['Express.js', 'Node.js'])"
    )
    skill_evidence: List[str] = Field(
        description="Brief review-based evidence snippets supporting the skill assessment"
    )

profile_parser = PydanticOutputParser(pydantic_object=ReviewerProfile)

skills_analysis_prompt = PromptTemplate(
    input_variables=["review_data", "reviewer_name"],
    template="""Analyze the following GitHub Review data for reviewer {reviewer_name} and extract their JavaScript technical skills and expertise.

Review Data:
{review_data}

You are NOT given any predefined skill taxonomy. Infer skills directly from the review evidence.

Look for evidence in PR titles, descriptions, labels, review states, review comments, and the types of changes being reviewed. Identify whatever JavaScript-related skills are actually demonstrated, such as:
- Frameworks, libraries, and runtimes (Express, React, Node.js, etc.)
- Language features and patterns (async/await, modules, prototypes, etc.)
- Backend, frontend, testing, DevOps, documentation, or tooling expertise
- Review focus areas visible in their feedback and approval patterns

For each skill you identify, note which review(s) or signals support it.

Provide a detailed, evidence-based analysis of this reviewer's JavaScript ecosystem skills.""",
    partial_variables={"format_instructions": "Provide a detailed JavaScript-focused technical analysis grounded in review evidence."}
)

profile_generation_prompt = PromptTemplate(
    input_variables=["skills_analysis", "reviewer_name", "review_count", "repo_list"],
    template="""Based on the following JavaScript skills analysis, generate a comprehensive JavaScript reviewer profile.

Reviewer: {reviewer_name}
Total Reviews: {review_count}
Repositories: {repo_list}

Skills Analysis:
{skills_analysis}

Generate a JavaScript reviewer profile that includes:
1. Experience level assessment (Junior/Mid/Senior/Lead) based on review quality and technical depth
2. Primary JavaScript technical skills and frameworks demonstrated in their reviews
3. Programming languages expertise
4. Overall JavaScript reviewer summary
5. identified_skills: your own skill taxonomy inferred from the review evidence
6. skill_evidence: short review-based evidence snippets supporting the assessment

========== RULES FOR identified_skills ==========
- Create your own meaningful category names based on what the reviews actually show
- Do NOT use any predefined KU/Knowledge Unit taxonomy
- Include specific technologies, patterns, and practices you can support from the review data
- Use empty arrays [] for categories with no supporting evidence
- Use plain skill names only — do NOT append counts, frequency, or numeric metadata
- Prefer concrete skills (e.g., "Express.js", "Middleware", "Code Review") over vague labels

Example identified_skills:
{{
  "Backend Frameworks": ["Express.js", "Node.js"],
  "Code Review & Quality": ["API design feedback", "middleware review"],
  "Dependency Management": ["npm", "semver upgrades"],
  "Testing": ["Jest"],
  "Asynchronous Programming": ["Promises", "async/await"]
}}

========== RULES FOR skill_evidence ==========
- Provide 3-8 short bullet-style strings
- Each item should reference a concrete review signal (PR title, label, review comment, or review state)
- Do not invent evidence that is not supported by the analysis

========== CRITICAL RULES FOR JSON OUTPUT ==========
1. Return ONLY raw valid JSON — no markdown code fences (no ```json)
2. Do NOT include comments (no // or /* */)
3. Use \\n for line breaks inside string values, not literal newlines
4. The summary field MUST be a plain string, not a nested JSON object
========== END JSON OUTPUT RULES ==========

{format_instructions}

Be specific and evidence-based. Only include skills and categories supported by the review analysis.""",
    partial_variables={
        "format_instructions": profile_parser.get_format_instructions(),
    }
)


In [7]:
def create_analysis_chains(llm):
    
    skills_chain = LLMChain(
        llm=llm,
        prompt=skills_analysis_prompt,
        verbose=False  # Changed from True to False
    )
    
    profile_chain = LLMChain(
        llm=llm,
        prompt=profile_generation_prompt,
        verbose=False  # Changed from True to False
    )
    
    return skills_chain, profile_chain

def prepare_review_data_for_analysis(reviewer_data, max_reviews=20):
    """Prepare review data for LLM analysis (limit size to avoid token limits)"""
    
    review_details = reviewer_data['review_details'][:max_reviews] 
    
    review_summary = []
    for review in review_details:
        summary = {
            'pr_title': review['title'][:100],
            'repo': review.get('repo', ''),
            'labels': review['labels'],
            'review_state': review['review_state'],
            'additions': review['additions'],
            'deletions': review['deletions'],
            'files_changed': review['changed_files_count']
        }
        
        if review['description'] and len(review['description']) < 500:
            summary['pr_description'] = review['description'][:300]
        
        if review['review_body'] and len(review['review_body']) < 500:
            summary['review_comment'] = review['review_body'][:300]
        
        review_summary.append(summary)
    
    return review_summary

if llm and reviewer_stats:
    print(f"🔧 Creating analysis chains for ALL reviewers...")
    
    skills_chain, profile_chain = create_analysis_chains(llm)
    
    eligible_reviewers = [
        (name, stats) for name, stats in reviewer_stats.items()
        if stats['total_reviews'] >= 2 and not name.endswith('[bot]') and 'bot' not in name.lower()
    ]
    
    eligible_reviewers.sort(key=lambda x: x[1]['total_reviews'], reverse=True)
    
    print(f"✅ Chains created successfully")
    print(f"👥 Found {len(eligible_reviewers)} eligible reviewers")
    print(f"📊 Will process reviewers with 2+ reviews (excluding bots)")
    
elif llm:
    print("⚠️  LLM configured but no reviewers found")
    print("Please check that reviewer data was loaded correctly")
else:
    print("⚠️  LLM not configured - please run the LLM configuration cell first")
    print("Set up your OpenRouter API key to continue")

🔧 Creating analysis chains for ALL reviewers...
✅ Chains created successfully
👥 Found 31 eligible reviewers
📊 Will process reviewers with 2+ reviews (excluding bots)


In [8]:
def clean_llm_json_response(text: str) -> str:
    text = text.strip()
    fence_match = re.search(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    if fence_match:
        text = fence_match.group(1).strip()
    elif text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text).strip()

    cleaned_lines = []
    for line in text.split('\n'):
        cleaned_lines.append(re.sub(r'//.*$', '', line))
    text = '\n'.join(cleaned_lines)
    text = re.sub(r',\s*([}\]])', r'\1', text)
    return text


def repair_multiline_json_strings(text: str) -> str:
    result = []
    in_string = False
    escape = False

    for char in text:
        if in_string:
            if escape:
                result.append(char)
                escape = False
            elif char == '\\':
                result.append(char)
                escape = True
            elif char == '"':
                in_string = False
                result.append(char)
            elif char in '\n\r':
                result.append('\\n')
            else:
                result.append(char)
        else:
            if char == '"':
                in_string = True
            result.append(char)

    return ''.join(result)


def normalize_profile_data(data: dict) -> dict:
    if "reviewer_name" not in data and "developer_name" in data:
        data["reviewer_name"] = data.pop("developer_name")

    summary = data.get("summary")
    if isinstance(summary, dict):
        data["summary"] = (
            summary.get("summary")
            or summary.get("text")
            or json.dumps(summary)
        )
    return data


def parse_reviewer_profile_response(profile_result: str) -> ReviewerProfile:
    cleaned = clean_llm_json_response(profile_result)
    candidates = [cleaned, repair_multiline_json_strings(cleaned)]

    last_error = None
    for candidate in candidates:
        try:
            return profile_parser.parse(candidate)
        except Exception as parse_error:
            last_error = parse_error
            try:
                data = normalize_profile_data(json.loads(candidate))
                return ReviewerProfile(**data)
            except Exception as json_error:
                last_error = json_error

    raise last_error


def generate_reviewer_profile(reviewer_name, reviewer_data, review_summary):
    try:
        skills_analysis_input = {
            "reviewer_name": reviewer_name,
            "review_data": json.dumps(review_summary, indent=2)
        }
        
        print(f"  🔍 Analyzing skills for {reviewer_name}...")
        skills_analysis_result = skills_chain.run(skills_analysis_input)
        
        profile_input = {
            "reviewer_name": reviewer_name,
            "skills_analysis": skills_analysis_result,
            "review_count": reviewer_data['total_reviews'],
            "repo_list": ', '.join(reviewer_data['repos'])
        }
        
        print(f"  🤖 Generating profile for {reviewer_name}...")
        profile_result = profile_chain.run(profile_input)
        
        try:
            parsed_profile = parse_reviewer_profile_response(profile_result)
            profile_dict = parsed_profile.dict()
        except Exception as parse_error:
            print(f"  ⚠️  Profile parsing failed for {reviewer_name}: {parse_error}")
            profile_dict = {
                "reviewer_name": reviewer_name,
                "raw_response": profile_result,
                "parsing_error": str(parse_error)
            }
        
        return {
            "success": True,
            "profile": profile_dict
            # Removed skills_analysis and raw_profile from return to reduce output
        }
        
    except Exception as e:
        print(f"  ❌ Error generating profile for {reviewer_name}: {e}")
        return {
            "success": False,
            "error": str(e),
            "reviewer": reviewer_name
        }

In [9]:
import time

def load_checkpoint(checkpoint_file="reviewer_profile_generation_checkpoint.json"):
    """Load the last checkpoint to resume processing"""
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            return json.load(f)
    return {"processed_reviewers": [], "profiles": {}}

def save_checkpoint(data, checkpoint_file="reviewer_profile_generation_checkpoint.json"):
    """Save checkpoint to resume processing later"""
    with open(checkpoint_file, 'w') as f:
        json.dump(data, f, indent=2)

def save_profile_to_file(reviewer_name, profile_data, timestamp):
    """Save individual profile to file"""
    output_dir = "Reviewers Profiles"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    filename = f"{reviewer_name}_reviewer_profile_{timestamp}.json"
    filepath = os.path.join(output_dir, filename)
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(profile_data, f, indent=2, ensure_ascii=False)
    
    print(f"💾 Saved profile to {filepath}")
    return filepath


def regenerate_single_reviewer_profile(reviewer_name, output_dir="Reviewers Profiles"):
    """Regenerate and export a profile for one reviewer."""
    if not (llm and reviewer_stats and 'skills_chain' in globals() and 'profile_chain' in globals()):
        raise RuntimeError("Run setup cells first (LLM, data load, chains)")

    if reviewer_name not in reviewer_stats:
        raise ValueError(f"{reviewer_name} not found in reviewer_stats")

    reviewer_data = reviewer_stats[reviewer_name]
    review_summary = prepare_review_data_for_analysis(reviewer_data)

    if not review_summary:
        raise ValueError(f"No review data available for {reviewer_name}")

    print(f"🔄 Regenerating profile for {reviewer_name}...")
    result = generate_reviewer_profile(reviewer_name, reviewer_data, review_summary)

    profile = result.get("profile", {})
    if profile.get("parsing_error"):
        print(f"⚠️  Parsing issue: {profile['parsing_error']}")
    else:
        print(f"✅ Profile parsed successfully ({profile.get('experience_level', 'Unknown')} level)")

    stats_without_details = {
        'total_reviews': reviewer_data['total_reviews'],
        'total_comments': reviewer_data['total_comments'],
        'total_prs': reviewer_data['total_prs'],
        'repos': reviewer_data['repos']
    }

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    profile_data = {
        "reviewer_name": reviewer_name,
        "generated_at": timestamp,
        "llm_provider": f"OpenRouter ({llm.model_name})",
        "source": "GitHub Review Data Analysis (LLM-inferred skills, no KU taxonomy)",
        "stats": stats_without_details,
        "profile": result["profile"]
    }

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    filename = f"{reviewer_name}_reviewer_profile_{timestamp}.json"
    filepath = os.path.join(output_dir, filename)

    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(profile_data, f, indent=2, ensure_ascii=False)

    print(f"💾 Saved: {filepath}")
    return result, filepath


def run_profile_generation_session(batch_size=15, delay_seconds=2):
    """Run profile generation in batches with checkpoint support"""
    global eligible_reviewers, skills_chain, profile_chain
    
    if not eligible_reviewers:
        print("❌ No eligible reviewers found for processing")
        return []
    
    # Load checkpoint
    checkpoint_data = load_checkpoint()
    processed_reviewers = set(checkpoint_data.get("processed_reviewers", []))
    all_profiles = checkpoint_data.get("profiles", {})
    
    # Find reviewers to process
    reviewers_to_process = [
        (name, data) for name, data in eligible_reviewers 
        if name not in processed_reviewers
    ]
    
    if not reviewers_to_process:
        print("✅ All eligible reviewers have already been processed!")
        return []
    
    # Take only batch_size reviewers
    batch_reviewers = reviewers_to_process[:batch_size]
    
    print(f"🚀 Starting batch processing of {len(batch_reviewers)} reviewers...")
    print(f"📊 {len(processed_reviewers)} already processed, {len(reviewers_to_process)} remaining")
    print(f"⏱️  Delay between profiles: {delay_seconds} seconds\n")
    
    batch_results = []
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    for i, (reviewer_name, reviewer_data) in enumerate(batch_reviewers, 1):
        print(f"\n🔄 Processing reviewer {i}/{len(batch_reviewers)}: {reviewer_name}")
        print(f"📈 Stats: {reviewer_data['total_reviews']} reviews, {reviewer_data['total_prs']} PRs")
        
        try:
            review_summary = prepare_review_data_for_analysis(reviewer_data)
            
            if not review_summary:
                print(f"  ⚠️  No review data available for {reviewer_name}")
                continue
            
            result = generate_reviewer_profile(reviewer_name, reviewer_data, review_summary)
            
            if result["success"]:
                # Create stats without review_details
                stats_without_details = {
                    'total_reviews': reviewer_data['total_reviews'],
                    'total_comments': reviewer_data['total_comments'],
                    'total_prs': reviewer_data['total_prs'],
                    'repos': reviewer_data['repos']
                }
                
                # Add stats to profile (simplified version without verbose analysis)
                profile_data = {
                    "reviewer_name": reviewer_name,
                    "generated_at": timestamp,
                    "llm_provider": f"OpenRouter ({llm.model_name})",
                    "source": "GitHub Review Data Analysis (LLM-inferred skills, no KU taxonomy)",
                    "stats": stats_without_details,
                    "profile": result["profile"]
                }
                
                # Save individual profile
                save_profile_to_file(reviewer_name, profile_data, timestamp)
                
                # Add to collection
                all_profiles[reviewer_name] = profile_data
                processed_reviewers.add(reviewer_name)
                
                # Add to batch results
                batch_results.append({
                    "reviewer": reviewer_name,
                    "result": result,
                    "profile_data": profile_data
                })
                
                print(f"  ✅ Profile generated successfully for {reviewer_name}")
                
                # Show brief summary only
                if "profile" in result and "experience_level" in result["profile"]:
                    experience = result["profile"]["experience_level"]
                    skills_count = len(result["profile"].get("primary_skills", []))
                    category_count = len(result["profile"].get("identified_skills", {}))
                    evidence_count = len(result["profile"].get("skill_evidence", []))
                    print(f"  📝 {experience} level, {skills_count} primary skills, {category_count} LLM categories, {evidence_count} evidence items")
                
            else:
                print(f"  ❌ Failed to generate profile for {reviewer_name}: {result.get('error', 'Unknown error')}")
            
            # Save checkpoint after each successful profile
            checkpoint_data = {
                "processed_reviewers": list(processed_reviewers),
                "profiles": all_profiles
            }
            save_checkpoint(checkpoint_data)
            
            # Rate limiting
            if i < len(batch_reviewers):  # Don't delay after the last one
                time.sleep(delay_seconds)
            
        except Exception as e:
            print(f"  💥 Unexpected error processing {reviewer_name}: {e}")
            continue
    
    print(f"\n🎉 Batch processing completed!")
    print(f"✅ Successfully processed {len(batch_results)} reviewers in this batch")
    print(f"📁 Individual profiles saved to 'Reviewers Profiles' directory")
    
    remaining_count = len(reviewers_to_process) - len(batch_reviewers)
    if remaining_count > 0:
        print(f"📋 {remaining_count} reviewers remaining for future batches")
    else:
        print(f"🎯 All eligible reviewers have been processed!")
    
    return batch_results

# Setup for batch processing
if 'eligible_reviewers' in locals() and eligible_reviewers:
    print(f"📊 Found {len(eligible_reviewers)} eligible reviewers for processing")
    print("🔧 Batch processing functions ready!")
    print("Run the batch processing cell below to start generating profiles")
else:
    print("❌ No eligible reviewers found for processing")
    print("Please ensure reviewer data is loaded and LLM is configured")

📊 Found 31 eligible reviewers for processing
🔧 Batch processing functions ready!
Run the batch processing cell below to start generating profiles


In [11]:
BATCH_SIZE = 30       # Number of reviewers to process in each batch (reduced for initial test)
DELAY_SECONDS = 2      # Delay between each profile generation


batch_results = run_profile_generation_session(
    batch_size=BATCH_SIZE, 
    delay_seconds=DELAY_SECONDS
)

if batch_results:
    print(f"\n📋 Quick Preview of This Batch:")
    for i, profile_data in enumerate(batch_results[:3], 1):
        reviewer_name = profile_data["reviewer"]
        profile = profile_data["result"]["profile"]
        if profile and isinstance(profile, dict):
            experience = profile.get('experience_level', 'Unknown')
            skills_count = len(profile.get('primary_skills', []))
            category_count = len(profile.get('identified_skills', {}))
            evidence_count = len(profile.get('skill_evidence', []))
            print(f"  {i}. {reviewer_name}: {experience} level, {skills_count} primary skills, {category_count} LLM categories, {evidence_count} evidence items")
        else:
            print(f"  {i}. {reviewer_name}: Profile generated (parsing may have failed)")
    
    if len(batch_results) > 3:
        print(f"  ... and {len(batch_results) - 3} more profiles generated")
else:
    print("📝 No new profiles generated in this batch")

🚀 Starting batch processing of 29 reviewers...
📊 2 already processed, 29 remaining
⏱️  Delay between profiles: 2 seconds


🔄 Processing reviewer 1/29: marvinhagemeister
📈 Stats: 177 reviews, 147 PRs
  🔍 Analyzing skills for marvinhagemeister...
  🤖 Generating profile for marvinhagemeister...
💾 Saved profile to Reviewers Profiles\marvinhagemeister_reviewer_profile_20260918_034607.json
  ✅ Profile generated successfully for marvinhagemeister
  📝 Senior level, 6 primary skills, 7 LLM categories, 8 evidence items

🔄 Processing reviewer 2/29: developit
📈 Stats: 33 reviews, 18 PRs
  🔍 Analyzing skills for developit...
  🤖 Generating profile for developit...
💾 Saved profile to Reviewers Profiles\developit_reviewer_profile_20260918_034607.json
  ✅ Profile generated successfully for developit
  📝 Senior level, 6 primary skills, 7 LLM categories, 8 evidence items

🔄 Processing reviewer 3/29: Thompson1985
📈 Stats: 20 reviews, 17 PRs
  🔍 Analyzing skills for Thompson1985...
  🤖 Generating profile 